# Comparing GFlowNet Objectives: TB vs DB vs FL-DB

This notebook compares three GFlowNet training objectives:

| Objective | Description | Key Feature |
|-----------|-------------|-------------|
| **TB** (Trajectory Balance) | Global partition function | Learns single `log Z` |
| **DB** (Detailed Balance) | Per-state flow | Learns `F(s)` for each state |
| **FL-DB** (Forward-Looking DB) | DB + intermediate rewards | Supports trajectory rewards |

## Loss Functions

**TB:** $(\\log Z + \\sum \\log P_F - \\log R - \\sum \\log P_B)^2$

**DB:** $\\sum_t (\\log F(s_t) + \\log P_F - \\log F(s_{t+1}) - \\log P_B)^2$

**FL-DB:** Same as DB but with $- \\log R(s_t, a_t, s_{t+1})$ for intermediate rewards

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from gfn import train, TrainingConfig, generate_greedy_trajectory
from gfn.reward import TargetMatchReward
from gfn.visualization import plot_training_curves, plot_flow_network, plot_reward_distribution

%matplotlib inline

## 1. Setup

In [ ]:
# Common configuration
SEED = 42
N_EPISODES = 10_000  # Reduce for faster comparison
N_HID = 32
LR = 3e-3

# Target sequences
target_sequences = [
    ['A', 'B', 'B', 'C'], 
    ['A', 'B', 'C', 'ε'], 
    ['C', 'A', 'C', 'C'], 
    ['C', 'B', 'A', 'ε'], 
    ['C', 'C', 'B', 'A'], 
    ['C', 'C', 'C', 'A'],
]

# Reward function
reward_fn = TargetMatchReward(target_sequences, r_min=0.1)

print(f"Training {N_EPISODES} episodes with {len(target_sequences)} targets")

## 2. Train with Different Objectives

In [ ]:
# Train with TB (Trajectory Balance)
config_tb = TrainingConfig(
    seed=SEED,
    n_hid_units=N_HID,
    n_episodes=N_EPISODES,
    learning_rate=LR,
    objective="TB",
)

result_tb = train(reward_fn, config_tb)
print(f"TB - Final Z: {result_tb.final_Z:.2f}")

In [ ]:
# Train with DB (Detailed Balance)
config_db = TrainingConfig(
    seed=SEED,
    n_hid_units=N_HID,
    n_episodes=N_EPISODES,
    learning_rate=LR,
    objective="DB",
)

result_db = train(reward_fn, config_db)
print(f"DB - Final Z: {result_db.final_Z:.2f}")

In [ ]:
# Train with FL-DB (Forward-Looking Detailed Balance)
config_fldb = TrainingConfig(
    seed=SEED,
    n_hid_units=N_HID,
    n_episodes=N_EPISODES,
    learning_rate=LR,
    objective="FLDB",
)

result_fldb = train(reward_fn, config_fldb)
print(f"FL-DB - Final Z: {result_fldb.final_Z:.2f}")

## 3. Compare Training Curves

In [ ]:
# Compare training losses
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss comparison
ax = axes[0]
ax.plot(result_tb.losses, label='TB', alpha=0.8)
ax.plot(result_db.losses, label='DB', alpha=0.8)
ax.plot(result_fldb.losses, label='FL-DB', alpha=0.8)
ax.set_yscale('log')
ax.set_xlabel('Update Step')
ax.set_ylabel('Loss')
ax.set_title('Training Loss Comparison')
ax.legend()
ax.grid(True, alpha=0.3)

# Z comparison
ax = axes[1]
ax.plot(np.exp(result_tb.logZs), label='TB', alpha=0.8)
ax.plot(np.exp(result_db.logZs), label='DB', alpha=0.8)
ax.plot(np.exp(result_fldb.logZs), label='FL-DB', alpha=0.8)
ax.set_xlabel('Update Step')
ax.set_ylabel('Estimated Z')
ax.set_title('Partition Function Estimate')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nFinal Z estimates:")
print(f"  TB:    {result_tb.final_Z:.2f}")
print(f"  DB:    {result_db.final_Z:.2f}")
print(f"  FL-DB: {result_fldb.final_Z:.2f}")

## 4. Compare Reward Distributions

In [ ]:
# Compare reward distributions
def get_target_hit_rate(result, reward_fn):
    """Calculate percentage of samples that hit a target."""
    rewards = [reward_fn(state[1]) for state in result.sampled_states]
    return sum(1 for r in rewards if r > 0.1) / len(rewards) * 100

print("Target hit rate (% samples reaching targets):")
print(f"  TB:    {get_target_hit_rate(result_tb, reward_fn):.1f}%")
print(f"  DB:    {get_target_hit_rate(result_db, reward_fn):.1f}%")
print(f"  FL-DB: {get_target_hit_rate(result_fldb, reward_fn):.1f}%")

# Show distributions
print("\n--- TB ---")
plot_reward_distribution(result_tb, reward_fn)
print("\n--- DB ---")
plot_reward_distribution(result_db, reward_fn)
print("\n--- FL-DB ---")
plot_reward_distribution(result_fldb, reward_fn)

## 5. Flow Networks

In [ ]:
# TB Flow Network
print("=== TB Flow Network ===")
fig = plot_flow_network(result_tb.model, target_sequences=target_sequences)
plt.show()

In [ ]:
# DB Flow Network
print("=== DB Flow Network ===")
fig = plot_flow_network(result_db.model, target_sequences=target_sequences)
plt.show()

In [ ]:
# FL-DB Flow Network
print("=== FL-DB Flow Network ===")
fig = plot_flow_network(result_fldb.model, target_sequences=target_sequences)
plt.show()

## Summary

| Metric | TB | DB | FL-DB |
|--------|----|----|-------|
| Loss convergence | Fast | Medium | Medium |
| Z estimation | Single param | From F(s₀) | From F(s₀) |
| Flexibility | Simple | Per-state flow | + Intermediate rewards |

**Recommendations:**
- **TB**: Best for simple problems, faster convergence
- **DB**: Better credit assignment, more parameters to learn
- **FL-DB**: Use when you have meaningful intermediate rewards